# Multivariate Calculus for Machine Learning

**Soft prerequisite** -- helpful background that will be reinforced in Sessions 1-2 (math warm-up).

## Who is this for?
You know single-variable calculus: derivatives, the chain rule, basic integrals. This notebook extends those ideas to functions of **multiple variables**, which is the setting of virtually every ML optimization problem.

## What you will cover
| Subtopic | Why it matters in ML |
|---|---|
| Partial derivatives | Measure how loss changes w.r.t. one parameter |
| Gradient vector | Points in the direction of steepest ascent |
| Gradient descent | The core optimization loop |
| Jacobian matrix | Generalizes the gradient to vector-valued functions |
| Multivariable chain rule | The math behind backpropagation |
| Contour & quiver plots | Visualize all of the above |

**Estimated time:** 30-45 minutes.

**Requirements:** `numpy`, `matplotlib` (both ship with Anaconda / Google Colab).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["figure.dpi"] = 100

---
## 1. Partial Derivatives

Given $f(x, y)$, the **partial derivative** with respect to $x$ is the ordinary derivative taken while treating $y$ as a constant:

$$\frac{\partial f}{\partial x} = \lim_{h \to 0} \frac{f(x+h,\, y) - f(x,\, y)}{h}$$

Likewise for $\frac{\partial f}{\partial y}$.

### Example

Let $f(x, y) = x^2 + 3xy + y^2$.

$$\frac{\partial f}{\partial x} = 2x + 3y, \qquad \frac{\partial f}{\partial y} = 3x + 2y$$

We can verify numerically with a finite-difference approximation.

In [ ]:
def f(x, y):
    return x**2 + 3*x*y + y**2

# Analytical partial derivatives
def df_dx(x, y):
    return 2*x + 3*y

def df_dy(x, y):
    return 3*x + 2*y

# Numerical check (finite differences)
h = 1e-7
x0, y0 = 1.0, 2.0

num_dx = (f(x0 + h, y0) - f(x0, y0)) / h
num_dy = (f(x0, y0 + h) - f(x0, y0)) / h

print(f"df/dx at ({x0}, {y0}):  analytical = {df_dx(x0, y0):.4f},  numerical = {num_dx:.4f}")
print(f"df/dy at ({x0}, {y0}):  analytical = {df_dy(x0, y0):.4f},  numerical = {num_dy:.4f}")

### Practice 1.1

Compute the partial derivatives of $g(x, y) = \sin(x)\, e^{-y^2}$ analytically, then verify numerically at the point $(\pi/4,\; 1)$.

In [ ]:
# YOUR CODE HERE

---
## 2. The Gradient Vector

The **gradient** of $f : \mathbb{R}^n \to \mathbb{R}$ collects all partial derivatives into a vector:

$$\nabla f = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \vdots \\ \frac{\partial f}{\partial x_n} \end{bmatrix}$$

### Key geometric fact

$\nabla f(\mathbf{x})$ points in the **direction of steepest ascent** of $f$ at $\mathbf{x}$, and its magnitude equals the rate of increase in that direction.

Consequently, $-\nabla f(\mathbf{x})$ points in the direction of **steepest descent** -- this is why gradient descent works.

### Example: gradient of a quadratic

In [ ]:
# f(x, y) = x^2 + 3*x*y + y^2  (same as before)
# grad f = [2x + 3y, 3x + 2y]

# Evaluate the gradient at several points
points = np.array([[1, 2], [-1, 1], [0, -2], [2, -1]])

for p in points:
    grad = np.array([df_dx(*p), df_dy(*p)])
    print(f"Point {p}  ->  grad = {grad},  |grad| = {np.linalg.norm(grad):.3f}")

### Practice 2.1

For $f(x, y) = x^2 + 4y^2$ (an elliptic paraboloid):
1. Write the gradient analytically.
2. Evaluate it at $(3, 1)$ and $(0, -2)$.
3. At which of those two points is $f$ increasing fastest?

In [ ]:
# YOUR CODE HERE

---
## 3. Gradient Descent Intuition

In ML we want to **minimize** a loss function $L(\boldsymbol{\theta})$ over parameters $\boldsymbol{\theta}$. Gradient descent repeats:

$$\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta\, \nabla L(\boldsymbol{\theta}_t)$$

where $\eta > 0$ is the **learning rate**.

Each step moves in the direction of steepest descent. If $\eta$ is too large we overshoot; too small and convergence is slow.

### Example: minimize $f(x,y) = x^2 + 4y^2$

In [ ]:
def f_ellipse(x, y):
    return x**2 + 4*y**2

def grad_ellipse(x, y):
    return np.array([2*x, 8*y])

# Run gradient descent
lr = 0.08
theta = np.array([4.0, 3.0])  # starting point
trajectory = [theta.copy()]

for _ in range(30):
    theta = theta - lr * grad_ellipse(*theta)
    trajectory.append(theta.copy())

trajectory = np.array(trajectory)

# Plot
xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-4, 4, 200))
zz = f_ellipse(xx, yy)

fig, ax = plt.subplots()
ax.contour(xx, yy, zz, levels=20, cmap="viridis")
ax.plot(trajectory[:, 0], trajectory[:, 1], "o-", color="red", markersize=3, linewidth=1)
ax.plot(trajectory[0, 0], trajectory[0, 1], "rs", markersize=8, label="start")
ax.plot(0, 0, "k*", markersize=12, label="minimum")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Gradient descent on $f(x,y) = x^2 + 4y^2$")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### Practice 3.1

Experiment with the learning rate:
1. Set `lr = 0.01`. How many more steps does it take to get close to $(0, 0)$?
2. Set `lr = 0.25`. What happens? (Hint: try it and look at the trajectory.)
3. Can you find a learning rate that converges in fewer than 15 steps?

In [ ]:
# YOUR CODE HERE

---
## 4. The Jacobian Matrix

The gradient applies to scalar-valued functions ($f : \mathbb{R}^n \to \mathbb{R}$). For a **vector-valued** function $\mathbf{F} : \mathbb{R}^n \to \mathbb{R}^m$, the analog is the **Jacobian matrix**:

$$\mathbf{J} = \begin{bmatrix} \frac{\partial F_1}{\partial x_1} & \cdots & \frac{\partial F_1}{\partial x_n} \\ \vdots & \ddots & \vdots \\ \frac{\partial F_m}{\partial x_1} & \cdots & \frac{\partial F_m}{\partial x_n} \end{bmatrix} \in \mathbb{R}^{m \times n}$$

Row $i$ of $\mathbf{J}$ is the gradient of $F_i$. When $m = 1$ the Jacobian reduces to the gradient (as a row vector).

**In ML:** every neural-network layer is a vector-to-vector mapping. Backpropagation multiplies Jacobians via the chain rule.

### Example

Let $\mathbf{F}(x, y) = \begin{bmatrix} x^2 y \\ 5x + \sin y \end{bmatrix}$.

$$\mathbf{J} = \begin{bmatrix} 2xy & x^2 \\ 5 & \cos y \end{bmatrix}$$

In [ ]:
def F(x, y):
    return np.array([x**2 * y, 5*x + np.sin(y)])

def jacobian_F(x, y):
    return np.array([
        [2*x*y,   x**2],
        [5,       np.cos(y)]
    ])

# Numerical Jacobian via finite differences
def numerical_jacobian(func, x, y, h=1e-7):
    f0 = func(x, y)
    m = len(f0)
    J = np.zeros((m, 2))
    for j, (dx, dy) in enumerate([(h, 0), (0, h)]):
        J[:, j] = (func(x + dx, y + dy) - f0) / h
    return J

x0, y0 = 1.0, np.pi / 4
print("Analytical Jacobian:")
print(jacobian_F(x0, y0))
print("\nNumerical Jacobian:")
print(numerical_jacobian(F, x0, y0))

### Practice 4.1

Compute the Jacobian of $\mathbf{G}(x, y, z) = \begin{bmatrix} x + 2y + 3z \\ xy \\ e^z \end{bmatrix}$ analytically. What is its shape? Verify numerically at $(1, 2, 0)$.

In [ ]:
# YOUR CODE HERE

---
## 5. The Chain Rule in Multiple Dimensions

In single-variable calculus: if $y = f(g(x))$, then $\frac{dy}{dx} = f'(g(x))\, g'(x)$.

The multivariable generalization replaces derivatives with Jacobians:

$$\mathbf{J}_{\mathbf{F} \circ \mathbf{G}} = \mathbf{J}_{\mathbf{F}} \cdot \mathbf{J}_{\mathbf{G}}$$

This is matrix multiplication of Jacobians.

**Why this matters for ML:** A neural network is a composition of layers: $\hat{y} = f_L \circ f_{L-1} \circ \cdots \circ f_1(\mathbf{x})$. Backpropagation computes the gradient of the loss by chaining Jacobians from the output layer back to the input -- that is the multivariable chain rule applied repeatedly.

### Example: two composed transformations

Let $\mathbf{u} = \mathbf{G}(x, y) = (x+y,\; xy)$ and $z = F(u_1, u_2) = u_1^2 + u_2$.

Then $\frac{\partial z}{\partial x}$ and $\frac{\partial z}{\partial y}$ can be found via:

$$\nabla_{(x,y)} z = \mathbf{J}_F \cdot \mathbf{J}_G$$

In [ ]:
# G: R^2 -> R^2
def G(x, y):
    return np.array([x + y, x * y])

# F: R^2 -> R
def F_scalar(u1, u2):
    return u1**2 + u2

# Composed: z(x,y) = F(G(x,y))
def z_composed(x, y):
    u = G(x, y)
    return F_scalar(u[0], u[1])

# Jacobians
def J_G(x, y):
    return np.array([[1, 1],     # d(x+y)/dx, d(x+y)/dy
                     [y, x]])    # d(xy)/dx,  d(xy)/dy

def J_F(u1, u2):
    return np.array([[2*u1, 1]])  # 1 x 2 (gradient as row vector)

x0, y0 = 2.0, 3.0
u = G(x0, y0)

# Chain rule: J_F @ J_G
chain = J_F(u[0], u[1]) @ J_G(x0, y0)
print(f"Chain rule gives dz/dx, dz/dy = {chain.flatten()}")

# Numerical verification
h = 1e-7
dz_dx_num = (z_composed(x0+h, y0) - z_composed(x0, y0)) / h
dz_dy_num = (z_composed(x0, y0+h) - z_composed(x0, y0)) / h
print(f"Numerical check:         dz/dx, dz/dy = [{dz_dx_num:.4f}, {dz_dy_num:.4f}]")

### Practice 5.1

Consider the composition that mimics a tiny neural network (no bias, no activation for simplicity):

- Layer 1: $\mathbf{h} = W_1 \mathbf{x}$, with $W_1 = \begin{bmatrix}1 & 2\\3 & 4\end{bmatrix}$ and $\mathbf{x} = \begin{bmatrix}x_1\\x_2\end{bmatrix}$.
- Layer 2: $y = \mathbf{w}_2^T \mathbf{h}$, with $\mathbf{w}_2 = \begin{bmatrix}1\\-1\end{bmatrix}$.

1. What are the Jacobians $\mathbf{J}_{\text{layer1}}$ and $\mathbf{J}_{\text{layer2}}$?
2. Use the chain rule to compute $\frac{\partial y}{\partial \mathbf{x}}$.
3. Verify numerically.

In [ ]:
# YOUR CODE HERE

---
## 6. Visualizing: Contour Plots with Gradient Arrows

A **contour plot** shows level curves $f(x,y) = c$. Overlaying **gradient arrows** (quiver plot) confirms that gradients are always perpendicular to contour lines and point toward increasing $f$.

### Example

In [ ]:
# Function: f(x,y) = sin(x) * cos(y)
xx, yy = np.meshgrid(np.linspace(-2*np.pi, 2*np.pi, 300),
                      np.linspace(-2*np.pi, 2*np.pi, 300))
zz = np.sin(xx) * np.cos(yy)

# Gradient on a coarser grid for arrows
xq, yq = np.meshgrid(np.linspace(-2*np.pi, 2*np.pi, 16),
                      np.linspace(-2*np.pi, 2*np.pi, 16))
u = np.cos(xq) * np.cos(yq)       # df/dx
v = -np.sin(xq) * np.sin(yq)      # df/dy

fig, ax = plt.subplots()
cs = ax.contourf(xx, yy, zz, levels=20, cmap="RdBu_r")
ax.contour(xx, yy, zz, levels=20, colors="k", linewidths=0.3)
ax.quiver(xq, yq, u, v, color="black", scale=25)
fig.colorbar(cs, ax=ax, label="f(x, y)")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title(r"$f(x,y) = \sin(x)\cos(y)$ with gradient field")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
# A second example: the elliptic paraboloid with gradient descent overlay
xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-4, 4, 200))
zz = xx**2 + 4*yy**2

xq, yq = np.meshgrid(np.linspace(-4.5, 4.5, 12), np.linspace(-3.5, 3.5, 10))
u = 2*xq    # df/dx
v = 8*yq    # df/dy

fig, ax = plt.subplots()
cs = ax.contour(xx, yy, zz, levels=15, cmap="viridis")
ax.clabel(cs, inline=True, fontsize=7)
# Show NEGATIVE gradient (descent direction)
ax.quiver(xq, yq, -u, -v, color="red", alpha=0.6, scale=120)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title(r"$f(x,y) = x^2 + 4y^2$ with $-\nabla f$ (descent directions)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### Practice 6.1

Create a contour plot with gradient arrows for $f(x, y) = (x - 1)^2 + (y + 2)^2$. Where is the minimum? Do the gradient arrows confirm this?

In [ ]:
# YOUR CODE HERE

### Practice 6.2

Run gradient descent on $f(x, y) = (x-1)^2 + (y+2)^2$ starting from $(5, 5)$ with $\eta = 0.1$ for 30 steps. Plot the trajectory on top of the contour plot from 6.1.

In [ ]:
# YOUR CODE HERE

---
## Self-Assessment

If you can answer these confidently, you are ready for Sessions 1-2.

1. Given $f(x, y, z) = x^2 y + e^{yz}$, write down $\nabla f$. What is its shape?

2. A loss function $L(\theta_1, \theta_2)$ has gradient $\nabla L = [4, -2]$ at the current parameters. If the learning rate is $0.1$, what are the new parameters after one gradient descent step?

3. You have two functions composed as $\mathbf{h} = g(\mathbf{x})$ then $y = f(\mathbf{h})$. The Jacobian of $g$ is $3 \times 2$ and the Jacobian of $f$ is $1 \times 3$. What is the shape of $\frac{\partial y}{\partial \mathbf{x}}$? Which multiplication gives it?

4. On a contour plot, the gradient at any point is (perpendicular / parallel / tangent) to the contour line through that point. Fill in the blank and explain why.

---
## References

1. **3Blue1Brown -- Essence of Calculus** (especially the multivariable episodes):  
   [https://www.3blue1brown.com/topics/calculus](https://www.3blue1brown.com/topics/calculus)

2. **3Blue1Brown -- Gradient descent, how neural networks learn** (Chapter 2 of the Deep Learning series):  
   [https://www.youtube.com/watch?v=IHZwWFHWa-w](https://www.youtube.com/watch?v=IHZwWFHWa-w)

3. **Khan Academy -- Multivariable Calculus** (partial derivatives, gradient, Jacobian):  
   [https://www.khanacademy.org/math/multivariable-calculus](https://www.khanacademy.org/math/multivariable-calculus)